In [ ]:
# brain_voxel_pytorch_with_manual_standardization.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pickle
import datetime
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset

# 导入自定义数据加载器
from BrainVoxel38PatientLoader import create_data_loaders, BrainVoxelDataset

# 设置随机种子以确保结果可重现
torch.manual_seed(42)
np.random.seed(42)

# 定义神经网络模型
class DenseNet4x4096(nn.Module):
    def __init__(self, input_dim=341, output_dim=102, dropout_rate=0.5, weight_decay=0.00001):
        super(DenseNet4x4096, self).__init__()
        self.weight_decay = weight_decay
        
        # 四个全连接层，每层4096个神经元，与原Keras模型一致
        self.fc1 = nn.Linear(input_dim, 4096)
        self.fc2 = nn.Linear(4096, 4096)
        self.fc3 = nn.Linear(4096, 4096)
        self.fc4 = nn.Linear(4096, 4096)
        self.fc5 = nn.Linear(4096, output_dim)
        
        # 激活函数和Dropout层
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.softmax = nn.Softmax(dim=1)
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        x = self.dropout(self.relu(self.fc4(x)))
        x = self.fc5(x)
        return x

def compute_l2_loss(model, weight_decay):
    """计算L2正则化损失"""
    l2_loss = 0
    for param in model.parameters():
        l2_loss += torch.sum(param ** 2)
    return weight_decay * l2_loss

def train_model(model, train_loader, valid_loader, device, num_epochs=25, 
                batch_size=128, learning_rate=0.00001, weight_decay=0.00001):
    """训练模型函数"""
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # 记录训练历史
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    # 训练循环
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # 将one-hot编码标签转换为类别索引 (PyTorch的CrossEntropyLoss要求类别索引而非one-hot编码)
            target_indices = torch.argmax(labels, dim=1)
            
            # 前向传播
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, target_indices)
            
            # 添加L2正则化损失
            l2_loss = compute_l2_loss(model, weight_decay)
            loss += l2_loss
            
            # 反向传播和优化
            loss.backward()
            optimizer.step()
            
            # 统计训练损失和准确率
            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == target_indices).sum().item()
        
        # 计算训练集平均损失和准确率
        train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        
        # 验证阶段
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # 将one-hot编码标签转换为类别索引
                target_indices = torch.argmax(labels, dim=1)
                
                # 前向传播
                outputs = model(inputs)
                loss = criterion(outputs, target_indices)
                
                # 统计验证损失和准确率
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == target_indices).sum().item()
        
        # 计算验证集平均损失和准确率
        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        
        # 更新历史记录
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # 打印本轮训练结果
        print(f'Epoch {epoch+1}/{num_epochs}, '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
    
    return history

def plot_training_history(history):
    """绘制训练历史曲线"""
    plt.figure(figsize=(12, 5))
    
    # 绘制损失曲线
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Loss over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # 绘制准确率曲线
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Training Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Accuracy over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.tight_layout()
    return plt

def analyze_data_distribution_sampled(data, title, sample_size=100000):
    """通过随机采样分析数据分布并绘制直方图"""
    # 随机采样
    np.random.seed(42)
    if len(data) > sample_size:
        indices = np.random.choice(len(data), sample_size, replace=False)
        sampled_data = data[indices]
    else:
        sampled_data = data
    
    # 计算基本统计信息 (基于原始数据，确保统计准确)
    mean_val = np.mean(data)
    std_val = np.std(data)
    min_val = np.min(data)
    max_val = np.max(data)
    
    # 打印统计信息
    print(f"{title} - 统计信息:")
    print(f"均值: {mean_val:.4f}, 标准差: {std_val:.4f}")
    print(f"最小值: {min_val:.4f}, 最大值: {max_val:.4f}")
    print(f"形状: {data.shape}, 采样: {sampled_data.shape}")
    
    # 绘制直方图 (基于采样数据)
    plt.figure(figsize=(10, 6))
    sns.histplot(sampled_data.reshape(-1), kde=True)
    plt.title(f"{title} 数据分布 (随机采样)")
    plt.xlabel("值")
    plt.ylabel("频率")
    plt.axvline(mean_val, color='r', linestyle='--', label=f'均值: {mean_val:.4f}')
    plt.axvline(mean_val + std_val, color='g', linestyle='--', label=f'均值+标准差: {mean_val+std_val:.4f}')
    plt.axvline(mean_val - std_val, color='g', linestyle='--', label=f'均值-标准差: {mean_val-std_val:.4f}')
    plt.legend()
    
    return plt
    

def analyze_feature_distributions(data, title, num_features=5):
    """分析特定数量特征的分布"""
    # 随机选择几个特征
    np.random.seed(42)
    selected_features = np.random.choice(data.shape[1], num_features, replace=False)
    
    # 创建图表
    fig, axes = plt.subplots(num_features, 1, figsize=(10, 4 * num_features))
    
    for i, feature_idx in enumerate(selected_features):
        feature_data = data[:, feature_idx]
        
        # 计算统计信息
        mean_val = np.mean(feature_data)
        std_val = np.std(feature_data)
        
        # 绘制直方图
        sns.histplot(feature_data, kde=True, ax=axes[i])
        axes[i].set_title(f"{title} - 特征 {feature_idx} 分布")
        axes[i].axvline(mean_val, color='r', linestyle='--', label=f'均值: {mean_val:.4f}')
        axes[i].axvline(mean_val + std_val, color='g', linestyle='--', label=f'均值+标准差: {mean_val+std_val:.4f}')
        axes[i].axvline(mean_val - std_val, color='g', linestyle='--', label=f'均值-标准差: {mean_val-std_val:.4f}')
        axes[i].legend()
    
    plt.tight_layout()
    return plt





In [ ]:
# 设置基础路径和超参数
base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/reorganized_fold_data'
export_path = './models/'
os.makedirs(export_path, exist_ok=True)

# 模型超参数 - 与原Keras模型保持一致
batch_size = 128
num_epochs = 25
learning_rate = 0.00001
weight_decay = 0.00001

# 检测可用设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 创建数据加载器 - 关闭内置标准化
print("加载数据...")
train_loader_raw, valid_loader_raw, test_loader_raw = create_data_loaders(
    base_dir=base_dir,
    batch_size=batch_size,
    test_patient_id=38,
    seed=42
)

# 提取原始数据用于标准化前后的分析
train_features_raw = []
train_labels_raw = []
valid_features_raw = []
valid_labels_raw = []
test_features_raw = []
test_labels_raw = []

# 从数据加载器中提取数据
for inputs, labels in train_loader_raw:
    train_features_raw.append(inputs.numpy())
    train_labels_raw.append(labels.numpy())

for inputs, labels in valid_loader_raw:
    valid_features_raw.append(inputs.numpy())
    valid_labels_raw.append(labels.numpy())

for inputs, labels in test_loader_raw:
    test_features_raw.append(inputs.numpy())
    test_labels_raw.append(labels.numpy())

# 合并批次数据
train_features_raw = np.vstack(train_features_raw)
train_labels_raw = np.vstack(train_labels_raw)
valid_features_raw = np.vstack(valid_features_raw)
valid_labels_raw = np.vstack(valid_labels_raw)
test_features_raw = np.vstack(test_features_raw)
test_labels_raw = np.vstack(test_labels_raw)

# 创建时间戳用于保存文件
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 分析标准化前的数据分布
print("\n=== 标准化前的数据分析 ===")

# 分析整体数据分布（英文标题）
plt_train_raw = analyze_data_distribution_sampled(train_features_raw, "Training Set Before Normalization")
plt_train_raw.savefig(f"{export_path}train_raw_distribution_{timestamp}.png")
plt_train_raw.close()

plt_valid_raw = analyze_data_distribution_sampled(valid_features_raw, "Validation Set Before Normalization")
plt_valid_raw.savefig(f"{export_path}valid_raw_distribution_{timestamp}.png")
plt_valid_raw.close()

plt_test_raw = analyze_data_distribution_sampled(test_features_raw, "Test Set Before Normalization")
plt_test_raw.savefig(f"{export_path}test_raw_distribution_{timestamp}.png")
plt_test_raw.close()

# 分析特定特征的分布
plt_train_features_raw = analyze_feature_distributions(train_features_raw, "Feature Distributions - Training Set (Before Normalization)")
plt_train_features_raw.savefig(f"{export_path}train_raw_feature_distributions_{timestamp}.png")
plt_train_features_raw.close()

# 应用标准化
print("\n应用标准化...")
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features_raw)
valid_features_scaled = scaler.transform(valid_features_raw)
test_features_scaled = scaler.transform(test_features_raw)

# 保存标准化器
scaler_filename = f"{export_path}scaler_{timestamp}.pkl"
with open(scaler_filename, 'wb') as f:
    pickle.dump(scaler, f)
print(f"标准化器已保存至: {scaler_filename}")

# 分析标准化后的数据分布
print("\n=== 标准化后的数据分析 ===")

plt_train_scaled = analyze_data_distribution_sampled(train_features_scaled, "Training Set After Normalization")
plt_train_scaled.savefig(f"{export_path}train_scaled_distribution_{timestamp}.png")
plt_train_scaled.close()

plt_valid_scaled = analyze_data_distribution_sampled(valid_features_scaled, "Validation Set After Normalization")
plt_valid_scaled.savefig(f"{export_path}valid_scaled_distribution_{timestamp}.png")
plt_valid_scaled.close()

plt_test_scaled = analyze_data_distribution_sampled(test_features_scaled, "Test Set After Normalization")
plt_test_scaled.savefig(f"{export_path}test_scaled_distribution_{timestamp}.png")
plt_test_scaled.close()

# 分析特定特征的分布
plt_train_features_scaled = analyze_feature_distributions(train_features_scaled, "Feature Distributions - Training Set (After Normalization)")
plt_train_features_scaled.savefig(f"{export_path}train_scaled_feature_distributions_{timestamp}.png")
plt_train_features_scaled.close()


In [ ]:

# 创建使用标准化数据的数据集和数据加载器
train_dataset = BrainVoxelDataset(train_features_scaled, train_labels_raw)
valid_dataset = BrainVoxelDataset(valid_features_scaled, valid_labels_raw)
test_dataset = BrainVoxelDataset(test_features_scaled, test_labels_raw)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 创建模型
print("\n创建模型...")
model = DenseNet4x4096(input_dim=341, output_dim=102, 
                        dropout_rate=0.5, weight_decay=weight_decay)
model = model.to(device)

# 训练模型
print("\n开始训练...")
history = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    device=device,
    num_epochs=num_epochs,
    batch_size=batch_size,
    learning_rate=learning_rate,
    weight_decay=weight_decay
)

# 保存模型
model_filename = f"{export_path}dense_4x4096_model_{timestamp}.pt"
torch.save(model.state_dict(), model_filename)
print(f"模型已保存至: {model_filename}")

# 绘制训练历史
plt = plot_training_history(history)
history_plot_filename = f"{export_path}training_history_{timestamp}.png"
plt.savefig(history_plot_filename)
print(f"训练历史图表已保存至: {history_plot_filename}")
plt.close()

# 模型评估
print("\n评估模型...")
model.eval()
test_loss = 0
test_correct = 0
test_total = 0
criterion = nn.CrossEntropyLoss()

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        target_indices = torch.argmax(labels, dim=1)
        
        outputs = model(inputs)
        loss = criterion(outputs, target_indices)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == target_indices).sum().item()

test_loss = test_loss / test_total
test_acc = test_correct / test_total

print(f"测试集结果 - 损失: {test_loss:.4f}, 准确率: {test_acc:.4f}")

print("\n完成所有操作!")

In [ ]:
# 绘制训练历史
plt = plot_training_history(history)
history_plot_filename = f"{export_path}training_history_{timestamp}.png"
plt.savefig(history_plot_filename)
print(f"训练历史图表已保存至: {history_plot_filename}")
plt.close()

# 模型评估
print("\n评估模型...")
model.eval()
test_loss = 0
test_correct = 0
test_total = 0
criterion = nn.CrossEntropyLoss()

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        target_indices = torch.argmax(labels, dim=1)
        
        outputs = model(inputs)
        loss = criterion(outputs, target_indices)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == target_indices).sum().item()

test_loss = test_loss / test_total
test_acc = test_correct / test_total

print(f"测试集结果 - 损失: {test_loss:.4f}, 准确率: {test_acc:.4f}")

print("\n完成所有操作!")
